In [1]:
# ============================================================
# CELL 1: SETUP & INSTALLATION
# ============================================================

# Mount Google Drive untuk menyimpan hasil
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q pytorch-transformers tensorboardX multiprocess pyrouge
!pip install -q transformers datasets evaluate rouge-score

# Clone PreSumm repository
%cd /content
!rm -rf PreSumm  # Hapus jika sudah ada
!git clone https://github.com/nlpyang/PreSumm.git

# Buat folder yang diperlukan
!mkdir -p /content/data_processed
!mkdir -p /content/bert_data
!mkdir -p /content/models
!mkdir -p /content/logs
!mkdir -p /content/results

print("✅ Setup complete!")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.5/60.5 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 129.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
/content
Cloning into 'PreSumm'...
remote: Enumerating objects: 423, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 423 (delta 1), reused 1 (delta 0), pack-reused 417 (from 1)
Receiving objec

In [2]:
# ============================================================
# CELL 2: LOAD DATASET
# ============================================================

from datasets import load_dataset

print("Loading DialogSum dataset...")
dataset = load_dataset('knkarthick/dialogsum')

print("\n✅ Dataset loaded!")
print(f"Train: {len(dataset['train'])} examples")
print(f"Validation: {len(dataset['validation'])} examples")
print(f"Test: {len(dataset['test'])} examples")

print("\n📋 Columns:", dataset['train'].column_names)

print("\n📝 Contoh data:")
example = dataset['train'][0]
print(f"Dialogue:\n{example['dialogue'][:500]}...")
print(f"\nSummary:\n{example['summary']}")

Loading DialogSum dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]


✅ Dataset loaded!
Train: 12460 examples
Validation: 500 examples
Test: 1500 examples

📋 Columns: ['id', 'dialogue', 'summary', 'topic']

📝 Contoh data:
Dialogue:
#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your ey...

Summary:
Mr. Smith's getting a check-up, and Doctor Hawkins advises him to have one every year. Hawkins'll give some information about their classes and medications to help Mr. Smith quit smoking.


In [3]:
# ============================================================
# CELL 3: PREPROCESSING KE JSON
# ============================================================

from transformers import BertTokenizer
import json
import os

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def preprocess_for_bertsum(dataset, output_dir, split='train'):
    """
    Konversi DialogSum ke format JSON untuk BertSum
    """
    os.makedirs(output_dir, exist_ok=True)

    processed_data = []
    skipped = 0

    for idx, example in enumerate(dataset[split]):
        try:
            source = example['dialogue']
            target = example['summary']

            # Skip jika kosong
            if not source or not target:
                skipped += 1
                continue

            # Tokenize
            src_tokens = tokenizer.tokenize(str(source))[:510]
            tgt_tokens = tokenizer.tokenize(str(target))[:128]

            data_point = {
                'src': src_tokens,
                'tgt': tgt_tokens,
                'src_txt': str(source),
                'tgt_txt': str(target)
            }
            processed_data.append(data_point)

        except Exception as e:
            skipped += 1
            continue

    # Save
    output_path = os.path.join(output_dir, f'{split}.json')
    with open(output_path, 'w') as f:
        for item in processed_data:
            f.write(json.dumps(item) + '\n')

    print(f"✅ {split}: Saved {len(processed_data)} examples (skipped {skipped})")
    return processed_data

# Jalankan preprocessing
output_dir = '/content/data_processed'

print("="*50)
print("PREPROCESSING DATASET")
print("="*50)

for split in ['train', 'validation', 'test']:
    preprocess_for_bertsum(dataset, output_dir, split)

print("\n📁 Files created:")
!ls -la /content/data_processed/

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

PREPROCESSING DATASET
✅ train: Saved 12460 examples (skipped 0)
✅ validation: Saved 500 examples (skipped 0)
✅ test: Saved 1500 examples (skipped 0)

📁 Files created:
total 37228
drwxr-xr-x 2 root root     4096 Jan 30 02:30 .
drwxr-xr-x 1 root root     4096 Jan 30 02:28 ..
-rw-r--r-- 1 root root  3919087 Jan 30 02:30 test.json
-rw-r--r-- 1 root root 32892167 Jan 30 02:30 train.json
-rw-r--r-- 1 root root  1294692 Jan 30 02:30 validation.json


In [4]:
# ============================================================
# CELL 4: CONVERT KE FORMAT PYTORCH (.pt) - FIXED VERSION
# ============================================================

import torch
import json
import os
import gc
import re
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def convert_to_bertsum_format(input_dir, output_dir, split):
    """
    Konversi JSON ke format PyTorch (.pt) untuk BertSum
    FIXED: Includes src_sent_labels yang diperlukan
    """
    os.makedirs(output_dir, exist_ok=True)

    input_file = os.path.join(input_dir, f'{split}.json')

    if not os.path.exists(input_file):
        print(f"❌ File {input_file} tidak ditemukan!")
        return 0

    datasets = []

    with open(input_file, 'r') as f:
        for idx, line in enumerate(f):
            try:
                data = json.loads(line.strip())

                src_txt = data['src_txt']
                tgt_txt = data['tgt_txt']

                # Split source menjadi kalimat
                src_sentences = [s.strip() for s in src_txt.replace('\r\n', '\n').split('\n') if s.strip()]

                # Jika tidak ada split yang bagus, split per kalimat
                if len(src_sentences) <= 1:
                    src_sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', src_txt) if s.strip()]

                if len(src_sentences) == 0:
                    src_sentences = [src_txt]

                # Tokenize setiap kalimat
                src_subtokens_all = []
                clss = []
                segs = []
                current_seg = 0

                for i, sent in enumerate(src_sentences):
                    # Tambah [CLS] di awal setiap kalimat
                    clss.append(len(src_subtokens_all))
                    src_subtokens_all.append('[CLS]')

                    # Tokenize kalimat
                    sent_tokens = tokenizer.tokenize(sent)
                    src_subtokens_all.extend(sent_tokens)

                    # Tambah [SEP] di akhir kalimat
                    src_subtokens_all.append('[SEP]')

                    # Segment IDs (alternating 0 dan 1)
                    sent_length = len(sent_tokens) + 2
                    segs.extend([current_seg] * sent_length)
                    current_seg = 1 - current_seg

                    # Batasi panjang maksimal
                    if len(src_subtokens_all) >= 510:
                        src_subtokens_all = src_subtokens_all[:510]
                        segs = segs[:510]
                        break

                # Convert tokens ke IDs
                src_subtoken_ids = tokenizer.convert_tokens_to_ids(src_subtokens_all)

                # Filter clss yang valid
                clss = [c for c in clss if c < len(src_subtoken_ids)]

                # ⭐ PENTING: src_sent_labels (diperlukan oleh BertSum)
                src_sent_labels = [0] * len(clss)
                if len(src_sent_labels) > 0:
                    src_sent_labels[0] = 1

                # Tokenize target dengan special tokens
                tgt_subtokens = ['[unused0]'] + tokenizer.tokenize(tgt_txt)[:126] + ['[unused1]']
                tgt_subtoken_ids = tokenizer.convert_tokens_to_ids(tgt_subtokens)

                # Validasi
                if len(src_subtoken_ids) < 5 or len(tgt_subtoken_ids) < 3:
                    continue
                if len(clss) == 0:
                    continue

                data_dict = {
                    'src': src_subtoken_ids,
                    'tgt': tgt_subtoken_ids,
                    'segs': segs,
                    'clss': clss,
                    'src_sent_labels': src_sent_labels,  # ⭐ WAJIB ADA
                    'src_txt': src_sentences,
                    'tgt_txt': tgt_txt
                }
                datasets.append(data_dict)

            except Exception as e:
                continue

            # Progress
            if (idx + 1) % 3000 == 0:
                print(f"  Processed {idx + 1} examples...")

    # Save
    output_file = os.path.join(output_dir, f'dialogsum.{split}.pt')
    torch.save(datasets, output_file)
    print(f"✅ {split}: Saved {len(datasets)} examples to {output_file}")

    gc.collect()
    return len(datasets)

# Hapus file lama
!rm -rf /content/bert_data/*

# Konversi
input_dir = '/content/data_processed'
output_dir = '/content/bert_data'

print("="*50)
print("CONVERTING TO BERTSUM FORMAT (.pt)")
print("="*50)

total = 0
for split in ['train', 'validation', 'test']:
    print(f"\nConverting {split}...")
    count = convert_to_bertsum_format(input_dir, output_dir, split)
    total += count

print(f"\n📊 Total: {total} examples converted")
print("\n📁 Files created:")
!ls -la /content/bert_data/

CONVERTING TO BERTSUM FORMAT (.pt)

Converting train...
  Processed 3000 examples...
  Processed 6000 examples...
  Processed 9000 examples...
  Processed 12000 examples...
✅ train: Saved 12460 examples to /content/bert_data/dialogsum.train.pt

Converting validation...
✅ validation: Saved 500 examples to /content/bert_data/dialogsum.validation.pt

Converting test...
✅ test: Saved 1500 examples to /content/bert_data/dialogsum.test.pt

📊 Total: 14460 examples converted

📁 Files created:
total 32224
drwxr-xr-x 2 root root     4096 Jan 30 02:31 .
drwxr-xr-x 1 root root     4096 Jan 30 02:28 ..
-rw-r--r-- 1 root root  3419927 Jan 30 02:31 dialogsum.test.pt
-rw-r--r-- 1 root root 28441245 Jan 30 02:31 dialogsum.train.pt
-rw-r--r-- 1 root root  1123771 Jan 30 02:31 dialogsum.validation.pt


In [5]:
# ============================================================
# CELL 5: VERIFIKASI FORMAT DATA
# ============================================================

import torch

print("="*50)
print("VERIFIKASI FORMAT DATA")
print("="*50)

data = torch.load('/content/bert_data/dialogsum.train.pt')
print(f"\n📊 Jumlah data training: {len(data)}")
print(f"\n🔑 Keys dalam data[0]: {list(data[0].keys())}")

# Cek required fields
required_fields = ['src', 'tgt', 'segs', 'clss', 'src_sent_labels', 'src_txt', 'tgt_txt']
print(f"\n✅ Checking required fields:")
all_present = True
for field in required_fields:
    status = "✅" if field in data[0] else "❌"
    print(f"   {status} {field}")
    if field not in data[0]:
        all_present = False

if all_present:
    print("\n🎉 Semua field yang diperlukan sudah ada!")
else:
    print("\n⚠️ Ada field yang missing! Cek preprocessing.")

# Detail contoh
print(f"\n📝 Detail data[0]:")
for key, value in data[0].items():
    if isinstance(value, list):
        if len(value) > 10:
            print(f"   {key}: {value[:10]}... (len={len(value)})")
        else:
            print(f"   {key}: {value}")
    else:
        print(f"   {key}: {value[:100] if isinstance(value, str) else value}...")

VERIFIKASI FORMAT DATA

📊 Jumlah data training: 12460

🔑 Keys dalam data[0]: ['src', 'tgt', 'segs', 'clss', 'src_sent_labels', 'src_txt', 'tgt_txt']

✅ Checking required fields:
   ✅ src
   ✅ tgt
   ✅ segs
   ✅ clss
   ✅ src_sent_labels
   ✅ src_txt
   ✅ tgt_txt

🎉 Semua field yang diperlukan sudah ada!

📝 Detail data[0]:
   src: [101, 1001, 2711, 2487, 1001, 1024, 7632, 1010, 2720, 1012]... (len=287)
   tgt: [1, 2720, 1012, 3044, 1005, 1055, 2893, 1037, 4638, 1011]... (len=43)
   segs: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]... (len=287)
   clss: [0, 25, 47, 75, 101, 139, 148, 182, 191, 218]... (len=12)
   src_sent_labels: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]... (len=12)
   src_txt: ["#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?", '#Person2#: I found it would be a good idea to get a check-up.', "#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.", '#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?', '#Pe

In [6]:
# ============================================================
# CELL 6: PATCH PRESUMM UNTUK PYTORCH 2.x COMPATIBILITY
# ============================================================

import os
import re

%cd /content/PreSumm/src

def patch_file(filepath, replacements):
    """Patch file dengan replacements yang diberikan"""
    if not os.path.exists(filepath):
        print(f"⚠️ File tidak ditemukan: {filepath}")
        return False

    with open(filepath, 'r') as f:
        content = f.read()

    original = content

    for old, new in replacements:
        content = content.replace(old, new)

    if content != original:
        with open(filepath, 'w') as f:
            f.write(content)
        print(f"✅ Patched: {filepath}")
        return True
    else:
        print(f"ℹ️ No changes needed: {filepath}")
        return False

print("="*50)
print("PATCHING PRESUMM FILES")
print("="*50)

# ========================================
# PATCH 1: Boolean operations (PyTorch 2.x)
# ========================================
print("\n📌 Patch 1: Boolean operations...")

boolean_replacements = [
    ('1 - (src == 0)', '(src != 0).long()'),
    ('1 - (tgt == 0)', '(tgt != 0).long()'),
    ('1 - (segs == 0)', '(segs != 0).long()'),
    ('1 - (clss == -1)', '(clss != -1).long()'),
]

patch_file('models/data_loader.py', boolean_replacements)

# ========================================
# PATCH 2: torch.load weights_only (PyTorch 2.6+)
# ========================================
print("\n📌 Patch 2: torch.load weights_only...")

files_to_patch = [
    'train.py',
    'train_abstractive.py',
    'train_extractive.py',
    'models/trainer.py',
    'models/model_builder.py',
    'models/predictor.py',
]

for filepath in files_to_patch:
    if not os.path.exists(filepath):
        continue

    with open(filepath, 'r') as f:
        content = f.read()

    # Skip jika sudah di-patch
    if 'weights_only=False' in content:
        print(f"ℹ️ Already patched: {filepath}")
        continue

    # Patch torch.load
    # Pattern 1: torch.load(path)
    content = re.sub(
        r'torch\.load$([^,$]+)\)',
        r'torch.load(\1, weights_only=False)',
        content
    )

    # Pattern 2: torch.load(path, map_location=xxx)
    content = re.sub(
        r'torch\.load$([^,]+),\s*map_location=([^$]+)\)',
        r'torch.load(\1, map_location=\2, weights_only=False)',
        content
    )

    # Fix double weights_only jika ada
    content = content.replace('weights_only=False, weights_only=False', 'weights_only=False')

    with open(filepath, 'w') as f:
        f.write(content)

    print(f"✅ Patched: {filepath}")

# ========================================
# VERIFIKASI PATCH
# ========================================
print("\n" + "="*50)
print("VERIFIKASI PATCH")
print("="*50)

print("\n📋 Boolean operations di data_loader.py:")
!grep -n "!= 0\|!= -1" models/data_loader.py | head -5

print("\n📋 torch.load di train_abstractive.py:")
!grep -n "torch.load" train_abstractive.py | head -3

print("\n✅ Patching complete!")

/content/PreSumm/src
PATCHING PRESUMM FILES

📌 Patch 1: Boolean operations...
✅ Patched: models/data_loader.py

📌 Patch 2: torch.load weights_only...
✅ Patched: train.py
✅ Patched: train_abstractive.py
✅ Patched: train_extractive.py
✅ Patched: models/trainer.py
✅ Patched: models/model_builder.py
✅ Patched: models/predictor.py

VERIFIKASI PATCH

📋 Boolean operations di data_loader.py:
33:            mask_src = (src != 0).long()
34:            mask_tgt = (tgt != 0).long()
39:            mask_cls = (clss != -1).long()

📋 torch.load di train_abstractive.py:
175:    checkpoint = torch.load(test_from, map_location=lambda storage, loc: storage)
208:    checkpoint = torch.load(test_from, map_location=lambda storage, loc: storage)
236:    checkpoint = torch.load(test_from, map_location=lambda storage, loc: storage)

✅ Patching complete!


In [7]:
# ============================================================
# CELL 7: TRAINING BERTSUMABS (OPTIMIZED HYPERPARAMETERS)
# ============================================================

%cd /content/PreSumm/src

print("="*50)
print("TRAINING BERTSUMABS")
print("="*50)
print("""
📊 Optimized Hyperparameters:
- lr_bert: 0.0005 (lebih stabil)
- lr_dec: 0.05 (mengurangi fluktuasi)
- train_steps: 8000 (lebih converge)
- warmup_steps: 1000 (smoother warmup)
- dec_dropout: 0.1 (optimal regularization)
- batch_size: 4, accum_count: 8 (effective batch = 32)
- label_smoothing: 0.1 (better generalization)
""")

# Training command dengan hyperparameter optimal
!python train.py \
    -task abs \
    -mode train \
    -bert_data_path /content/bert_data/dialogsum \
    -dec_dropout 0.1 \
    -model_path /content/models \
    -sep_optim true \
    -lr_bert 0.0005 \
    -lr_dec 0.05 \
    -save_checkpoint_steps 1000 \
    -batch_size 4 \
    -train_steps 8000 \
    -report_every 200 \
    -accum_count 8 \
    -use_bert_emb true \
    -use_interval true \
    -warmup_steps_bert 1000 \
    -warmup_steps_dec 1000 \
    -max_pos 512 \
    -visible_gpus 0 \
    -log_file /content/logs/train_abs.log \
    -label_smoothing 0.1

print("\n✅ Training complete!")
print("\n📁 Checkpoints saved:")
!ls -la /content/models/*.pt

/content/PreSumm/src
TRAINING BERTSUMABS

📊 Optimized Hyperparameters:
- lr_bert: 0.0005 (lebih stabil)
- lr_dec: 0.05 (mengurangi fluktuasi)
- train_steps: 8000 (lebih converge)
- warmup_steps: 1000 (smoother warmup)
- dec_dropout: 0.1 (optimal regularization)
- batch_size: 4, accum_count: 8 (effective batch = 32)
- label_smoothing: 0.1 (better generalization)

/content/PreSumm/src/others/pyrouge.py:73: SyntaxWarning: invalid escape sequence '\d'
  rouge.system_filename_pattern = 'SL.P.10.R.11.SL062003-(\d+).html'
/content/PreSumm/src/others/pyrouge.py:164: SyntaxWarning: invalid escape sequence '\d'
  E.g. "SL.P.10.R.11.SL062003-(\d+).html" will match the system
/content/PreSumm/src/others/pyrouge.py:189: SyntaxWarning: invalid escape sequence '\d'
  matched by the "(\d+)" part of the system filename pattern.
[2026-01-30 02:31:33,841 INFO] Namespace(task='abs', encoder='bert', mode='train', bert_data_path='/content/bert_data/dialogsum', model_path='/content/models', result_path='../r

In [8]:
# ============================================================
# CELL 8: CEK CHECKPOINT & TRAINING LOG
# ============================================================

print("="*50)
print("CHECKPOINTS & TRAINING LOG")
print("="*50)

print("\n📁 Model Checkpoints:")
!ls -la /content/models/*.pt 2>/dev/null || echo "No checkpoints found"

print("\n📋 Training Log (last 50 lines):")
!tail -50 /content/logs/train_abs.log 2>/dev/null || echo "Log file not found"

CHECKPOINTS & TRAINING LOG

📁 Model Checkpoints:
-rw-r--r-- 1 root root 2323813970 Jan 30 02:44 /content/models/model_step_1000.pt
-rw-r--r-- 1 root root 2323813970 Jan 30 02:57 /content/models/model_step_2000.pt
-rw-r--r-- 1 root root 2323813970 Jan 30 03:09 /content/models/model_step_3000.pt
-rw-r--r-- 1 root root 2323813970 Jan 30 03:22 /content/models/model_step_4000.pt
-rw-r--r-- 1 root root 2323813970 Jan 30 03:34 /content/models/model_step_5000.pt
-rw-r--r-- 1 root root 2323813970 Jan 30 03:46 /content/models/model_step_6000.pt
-rw-r--r-- 1 root root 2323813970 Jan 30 03:59 /content/models/model_step_7000.pt
-rw-r--r-- 1 root root 2323813970 Jan 30 04:12 /content/models/model_step_8000.pt

📋 Training Log (last 50 lines):
[2026-01-30 02:44:14,408 INFO] Step 1000/ 8000; acc:  36.44; ppl: 29.23; xent: 3.38; lr: 0.00001581;   0/387 tok/s;    719 sec
[2026-01-30 02:44:14,413 INFO] Saving checkpoint /content/models/model_step_1000.pt
[2026-01-30 02:47:05,737 INFO] Step 1200/ 8000; acc

In [15]:
# ============================================================
# FULL FIX - COPY PASTE SATU CELL INI
# ============================================================

%cd /content/PreSumm/src

import os
import re

def add_weights_only_to_torch_load(content):
    """Add weights_only=False to all torch.load calls"""

    # Hapus weights_only=False yang sudah ada untuk avoid duplicates
    content = content.replace(', weights_only=False', '')
    content = content.replace(',weights_only=False', '')

    # Function untuk menambah weights_only=False
    def replacer(match):
        s = match.group(0)
        # Hapus ) di akhir, tambah parameter, lalu tambah ) lagi
        return s[:-1] + ', weights_only=False)'

    # Apply ke semua torch.load(...)
    content = re.sub(r'torch\.load$[^)]+$', replacer, content)

    return content

# Files to patch
files = [
    'train.py',
    'train_abstractive.py',
    'train_extractive.py',
    'models/trainer.py',
    'models/model_builder.py',
    'models/predictor.py',
    'models/data_loader.py',
]

print("="*60)
print("PATCHING ALL FILES")
print("="*60)

for filepath in files:
    if not os.path.exists(filepath):
        print(f"⚠️ Skip (not found): {filepath}")
        continue

    with open(filepath, 'r') as f:
        content = f.read()

    new_content = add_weights_only_to_torch_load(content)

    with open(filepath, 'w') as f:
        f.write(new_content)

    print(f"✅ Patched: {filepath}")

# Verifikasi
print("\n" + "="*60)
print("VERIFIKASI - Semua torch.load harus punya weights_only=False")
print("="*60)

for filepath in files:
    if os.path.exists(filepath):
        result = os.popen(f'grep -c "torch.load" {filepath}').read().strip()
        result_with_param = os.popen(f'grep -c "weights_only=False" {filepath}').read().strip()
        print(f"{filepath}: {result} torch.load calls, {result_with_param} with weights_only=False")

print("\n✅ Patch complete! Sekarang jalankan generate summary lagi.")

/content/PreSumm/src
PATCHING ALL FILES
✅ Patched: train.py
✅ Patched: train_abstractive.py
✅ Patched: train_extractive.py
✅ Patched: models/trainer.py
✅ Patched: models/model_builder.py
✅ Patched: models/predictor.py
✅ Patched: models/data_loader.py

VERIFIKASI - Semua torch.load harus punya weights_only=False
train.py: 0 torch.load calls, 0 with weights_only=False
train_abstractive.py: 5 torch.load calls, 0 with weights_only=False
train_extractive.py: 3 torch.load calls, 0 with weights_only=False
models/trainer.py: 0 torch.load calls, 0 with weights_only=False
models/model_builder.py: 0 torch.load calls, 0 with weights_only=False
models/predictor.py: 0 torch.load calls, 0 with weights_only=False
models/data_loader.py: 1 torch.load calls, 0 with weights_only=False

✅ Patch complete! Sekarang jalankan generate summary lagi.


In [18]:
# ============================================================
# FULL FIX - SATU CELL LENGKAP
# ============================================================

%cd /content/PreSumm/src

print("="*60)
print("FIXING torch.load FOR PYTORCH 2.6+")
print("="*60)

# Files dan patterns yang perlu di-fix
files_and_patterns = {
    'train_abstractive.py': [
        ('torch.load(test_from, map_location=lambda storage, loc: storage)',
         'torch.load(test_from, map_location=lambda storage, loc: storage, weights_only=False)'),
        ('torch.load(args.test_from, map_location=lambda storage, loc: storage)',
         'torch.load(args.test_from, map_location=lambda storage, loc: storage, weights_only=False)'),
        ('torch.load(pt_file, map_location=lambda storage, loc: storage)',
         'torch.load(pt_file, map_location=lambda storage, loc: storage, weights_only=False)'),
    ],
    'train_extractive.py': [
        ('torch.load(test_from, map_location=lambda storage, loc: storage)',
         'torch.load(test_from, map_location=lambda storage, loc: storage, weights_only=False)'),
        ('torch.load(pt_file, map_location=lambda storage, loc: storage)',
         'torch.load(pt_file, map_location=lambda storage, loc: storage, weights_only=False)'),
    ],
    'models/data_loader.py': [
        ('torch.load(pts[0])', 'torch.load(pts[0], weights_only=False)'),
    ],
    'models/trainer.py': [
        ('torch.load(checkpoint_path, map_location=lambda storage, loc: storage)',
         'torch.load(checkpoint_path, map_location=lambda storage, loc: storage, weights_only=False)'),
    ],
}

import os

for filepath, patterns in files_and_patterns.items():
    if not os.path.exists(filepath):
        print(f"⚠️ Skip: {filepath} not found")
        continue

    with open(filepath, 'r') as f:
        content = f.read()

    changed = False
    for old, new in patterns:
        if old in content and new not in content:
            content = content.replace(old, new)
            changed = True

    if changed:
        with open(filepath, 'w') as f:
            f.write(content)
        print(f"✅ Fixed: {filepath}")
    else:
        print(f"ℹ️ No changes needed: {filepath}")

# Verifikasi
print("\n" + "="*60)
print("VERIFIKASI")
print("="*60)
print("\n📋 Semua torch.load dengan weights_only=False:")
!grep -rn "weights_only=False" train_abstractive.py train_extractive.py models/data_loader.py models/trainer.py 2>/dev/null

print("\n" + "="*60)
print("GENERATE SUMMARY")
print("="*60)

import glob
checkpoints = glob.glob('/content/models/model_step_*.pt')
checkpoints.sort(key=lambda x: int(x.split('_step_')[1].split('.pt')[0]))
best_checkpoint = checkpoints[-1] if checkpoints else None

print(f"\n📌 Using: {best_checkpoint}")

!rm -rf /content/results/*
!mkdir -p /content/results

!python train.py \
    -task abs \
    -mode test \
    -batch_size 4 \
    -test_batch_size 4 \
    -bert_data_path /content/bert_data/dialogsum \
    -log_file /content/logs/test_abs.log \
    -test_from {best_checkpoint} \
    -sep_optim true \
    -use_interval true \
    -visible_gpus 0 \
    -max_pos 512 \
    -max_length 100 \
    -alpha 0.95 \
    -min_length 20 \
    -result_path /content/results/

print("\n📁 Results:")
!ls -la /content/results/

/content/PreSumm/src
FIXING torch.load FOR PYTORCH 2.6+
✅ Fixed: train_abstractive.py
✅ Fixed: train_extractive.py
ℹ️ No changes needed: models/data_loader.py
ℹ️ No changes needed: models/trainer.py

VERIFIKASI

📋 Semua torch.load dengan weights_only=False:
train_abstractive.py:175:    checkpoint = torch.load(test_from, map_location=lambda storage, loc: storage, weights_only=False)
train_abstractive.py:208:    checkpoint = torch.load(test_from, map_location=lambda storage, loc: storage, weights_only=False)
train_abstractive.py:236:    checkpoint = torch.load(test_from, map_location=lambda storage, loc: storage, weights_only=False)
train_extractive.py:158:    checkpoint = torch.load(test_from, map_location=lambda storage, loc: storage, weights_only=False)
train_extractive.py:183:    checkpoint = torch.load(test_from, map_location=lambda storage, loc: storage, weights_only=False)

GENERATE SUMMARY

📌 Using: /content/models/model_step_8000.pt
[2026-01-30 04:27:19,525 INFO] Loading checkpo

In [19]:
# ============================================================
# FIX PREDICTOR.PY - INDEX_SELECT ERROR
# ============================================================

%cd /content/PreSumm/src

print("="*60)
print("FIXING predictor.py FOR PYTORCH 2.x")
print("="*60)

filepath = 'models/predictor.py'

with open(filepath, 'r') as f:
    content = f.read()

# Fix 1: index_select requires Long tensor indices
# Tambahkan .long() pada select_indices

content = content.replace(
    'alive_seq.index_select(0, select_indices)',
    'alive_seq.index_select(0, select_indices.long())'
)

content = content.replace(
    'alive_attn.index_select(0, select_indices)',
    'alive_attn.index_select(0, select_indices.long())'
)

content = content.replace(
    'src.index_select(0, select_indices)',
    'src.index_select(0, select_indices.long())'
)

# Fix untuk pattern lain yang mungkin ada
content = content.replace(
    '.index_select(0, select_indices)',
    '.index_select(0, select_indices.long())'
)

# Fix topk indices juga harus long
content = content.replace(
    'topk_ids = topk_ids.view(-1)',
    'topk_ids = topk_ids.view(-1).long()'
)

# Fix batch index
content = content.replace(
    'batch_index = batch_index.view(-1)',
    'batch_index = batch_index.view(-1).long()'
)

with open(filepath, 'w') as f:
    f.write(content)

print(f"✅ Fixed: {filepath}")

# Verifikasi
print("\n📋 Verifikasi changes:")
!grep -n "\.long()" models/predictor.py | head -10

/content/PreSumm/src
FIXING predictor.py FOR PYTORCH 2.x
✅ Fixed: models/predictor.py

📋 Verifikasi changes:
331:                [alive_seq.index_select(0, select_indices.long()),
372:            src_features = src_features.index_select(0, select_indices.long())


In [20]:
# ============================================================
# FIX KOMPREHENSIF - SEMUA INDEX OPERATIONS
# ============================================================

%cd /content/PreSumm/src

filepath = 'models/predictor.py'

with open(filepath, 'r') as f:
    lines = f.readlines()

new_lines = []
for i, line in enumerate(lines):
    # Jika ada index_select dan belum ada .long()
    if 'index_select' in line and '.long()' not in line:
        # Cari pattern: index_select(dim, indices)
        # Tambahkan .long() pada indices
        import re
        # Replace indices variable dengan indices.long()
        line = re.sub(
            r'\.index_select$(\d+),\s*(\w+)$',
            r'.index_select(\1, \2.long())',
            line
        )
    new_lines.append(line)

with open(filepath, 'w') as f:
    f.writelines(new_lines)

print("✅ Fixed all index_select in predictor.py")

# Verifikasi
print("\n📋 Semua index_select:")
!grep -n "index_select" models/predictor.py

/content/PreSumm/src
✅ Fixed all index_select in predictor.py

📋 Semua index_select:
331:                [alive_seq.index_select(0, select_indices.long()),
365:                topk_log_probs = topk_log_probs.index_select(0, non_finished)
366:                batch_index = batch_index.index_select(0, non_finished)
367:                batch_offset = batch_offset.index_select(0, non_finished)
368:                alive_seq = predictions.index_select(0, non_finished) \
372:            src_features = src_features.index_select(0, select_indices.long())
374:                lambda state, dim: state.index_select(dim, select_indices))


In [23]:
# ============================================================
# FULL FIX PREDICTOR.PY - SEMUA INDEX OPERATIONS
# ============================================================

%cd /content/PreSumm/src

print("="*60)
print("COMPREHENSIVE FIX FOR predictor.py")
print("="*60)

filepath = 'models/predictor.py'

with open(filepath, 'r') as f:
    content = f.read()

# ========================================
# FIX 1: Lambda function dengan select_indices
# ========================================
content = content.replace(
    'lambda state, dim: state.index_select(dim, select_indices)',
    'lambda state, dim: state.index_select(dim, select_indices.long())'
)

# ========================================
# FIX 2: index_select dengan non_finished
# ========================================
content = content.replace(
    '.index_select(0, non_finished)',
    '.index_select(0, non_finished.long())'
)

# ========================================
# FIX 3: Semua index_select lainnya
# ========================================
content = content.replace(
    '.index_select(0, select_indices)',
    '.index_select(0, select_indices.long())'
)

# ========================================
# FIX 4: Hindari double .long().long()
# ========================================
content = content.replace('.long().long()', '.long()')

with open(filepath, 'w') as f:
    f.write(content)

print("✅ All fixes applied to predictor.py")

# Verifikasi
print("\n📋 Verifikasi - semua index_select harus punya .long():")
!grep -n "index_select" models/predictor.py

# ========================================
# GENERATE SUMMARY
# ========================================
print("\n" + "="*60)
print("GENERATING SUMMARIES")
print("="*60)

import glob
checkpoints = glob.glob('/content/models/model_step_*.pt')
checkpoints.sort(key=lambda x: int(x.split('_step_')[1].split('.pt')[0]))
best_checkpoint = checkpoints[-1] if checkpoints else None

print(f"\n📌 Using: {best_checkpoint}")

!rm -rf /content/results/*
!mkdir -p /content/results

!python train.py \
    -task abs \
    -mode test \
    -batch_size 4 \
    -test_batch_size 4 \
    -bert_data_path /content/bert_data/dialogsum \
    -log_file /content/logs/test_abs.log \
    -test_from {best_checkpoint} \
    -sep_optim true \
    -use_interval true \
    -visible_gpus 0 \
    -max_pos 512 \
    -max_length 100 \
    -alpha 0.95 \
    -min_length 20 \
    -result_path /content/results/

print("\n" + "="*60)
print("RESULTS")
print("="*60)

print("\n📁 Files:")
!ls -la /content/results/

print("\n📋 Generated Summaries (first 5):")
!head -5 /content/results/*.candidate 2>/dev/null || echo "No results yet"

print("\n📋 Reference Summaries (first 5):")
!head -5 /content/results/*.gold 2>/dev/null || echo "No gold yet"

/content/PreSumm/src
COMPREHENSIVE FIX FOR predictor.py
✅ All fixes applied to predictor.py

📋 Verifikasi - semua index_select harus punya .long():
331:                [alive_seq.index_select(0, select_indices.long()),
365:                topk_log_probs = topk_log_probs.index_select(0, non_finished.long())
366:                batch_index = batch_index.index_select(0, non_finished.long())
367:                batch_offset = batch_offset.index_select(0, non_finished.long())
368:                alive_seq = predictions.index_select(0, non_finished.long()) \
372:            src_features = src_features.index_select(0, select_indices.long())
374:                lambda state, dim: state.index_select(dim, select_indices.long()))

GENERATING SUMMARIES

📌 Using: /content/models/model_step_8000.pt
[2026-01-30 04:35:05,346 INFO] Loading checkpoint from /content/models/model_step_8000.pt
Namespace(task='abs', encoder='bert', mode='test', bert_data_path='/content/bert_data/dialogsum', model_path='../m

In [25]:
# ============================================================
# LIHAT HASIL SUMMARY
# ============================================================

%cd /content/results

print("="*60)
print("HASIL GENERATED SUMMARIES")
print("="*60)

# Rename file agar mudah diakses (hapus titik di depan)
!mv .8000.candidate 8000.candidate 2>/dev/null
!mv .8000.gold 8000.gold 2>/dev/null
!mv .8000.raw_src 8000.raw_src 2>/dev/null

print("\n📁 Files:")
!ls -la

# Baca hasil
print("\n" + "="*60)
print("📋 GENERATED SUMMARIES (10 contoh pertama)")
print("="*60)
!head -10 8000.candidate

print("\n" + "="*60)
print("📋 REFERENCE SUMMARIES (10 contoh pertama)")
print("="*60)
!head -10 8000.gold

/content/results
HASIL GENERATED SUMMARIES

📁 Files:
total 1708
drwxr-xr-x 2 root root    4096 Jan 30 04:55 .
drwxr-xr-x 1 root root    4096 Jan 30 02:28 ..
-rw-r--r-- 1 root root  119448 Jan 30 04:50 8000.candidate
-rw-r--r-- 1 root root  173282 Jan 30 04:50 8000.gold
-rw-r--r-- 1 root root 1432893 Jan 30 04:50 8000.raw_src

📋 GENERATED SUMMARIES (10 contoh pertama)
# person1 # suggests # and # person2 # person2 about the traffic . # person1 ' s car .
# person1 # suggests # and # person2 # person2 about the traffic . # person1 ' s car .
# person2 # tells # person1 # person2 about # person1 . they need person2 # ' t to send .
# person1 # suggests # and # person2 # person2 about the traffic . # person1 ' s car .
# person2 # tells # person1 # person2 about # person1 . they need person2 # ' t to send .
# person2 # tells # person1 # person2 about # person1 . they need person2 # ' t to send .
# person1 # invites # and # person2 # ' s birthday party party party . they will celebrate .
# pers

In [26]:
# ============================================================
# BANDINGKAN HASIL
# ============================================================

print("="*60)
print("PERBANDINGAN GENERATED vs REFERENCE")
print("="*60)

# Baca file
with open('/content/results/8000.candidate', 'r') as f:
    candidates = [line.strip() for line in f.readlines() if line.strip()]

with open('/content/results/8000.gold', 'r') as f:
    references = [line.strip() for line in f.readlines() if line.strip()]

print(f"\n📊 Total Generated: {len(candidates)}")
print(f"📊 Total Reference: {len(references)}")

# Tampilkan 5 contoh perbandingan
num_examples = min(5, len(candidates), len(references))

for i in range(num_examples):
    print(f"\n{'='*70}")
    print(f"📝 Example {i+1}")
    print(f"{'='*70}")
    print(f"🤖 Generated:\n   {candidates[i]}")
    print(f"\n✅ Reference:\n   {references[i]}")

PERBANDINGAN GENERATED vs REFERENCE

📊 Total Generated: 1500
📊 Total Reference: 1500

📝 Example 1
🤖 Generated:
   # person1 # suggests # and # person2 # person2 about the traffic . # person1 ' s car .

✅ Reference:
   #Person2# decides to follow #Person1#'s suggestions on quitting driving to work and will try to use public transportations.

📝 Example 2
🤖 Generated:
   # person1 # suggests # and # person2 # person2 about the traffic . # person1 ' s car .

✅ Reference:
   #Person2# complains to #Person1# about the traffic jam, #Person1# suggests quitting driving and taking public transportation instead.

📝 Example 3
🤖 Generated:
   # person2 # tells # person1 # person2 about # person1 . they need person2 # ' t to send .

✅ Reference:
   Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.

📝 Example 4
🤖 Generated:
   # person1 # suggests # and # person2 # person2 about the traff

In [24]:
# ============================================================
# CELL 10: LIHAT HASIL SUMMARY
# ============================================================

import os

print("="*50)
print("HASIL GENERATED SUMMARIES")
print("="*50)

result_dir = '/content/results/'

# Cari file candidate dan gold
candidate_files = [f for f in os.listdir(result_dir) if 'candidate' in f] if os.path.exists(result_dir) else []
gold_files = [f for f in os.listdir(result_dir) if 'gold' in f] if os.path.exists(result_dir) else []

if candidate_files and gold_files:
    # Baca file
    with open(os.path.join(result_dir, candidate_files[0]), 'r') as f:
        candidates = f.readlines()

    with open(os.path.join(result_dir, gold_files[0]), 'r') as f:
        golds = f.readlines()

    print(f"\n📊 Total examples: {len(candidates)}")

    # Tampilkan 5 contoh
    num_examples = min(5, len(candidates))
    for i in range(num_examples):
        print(f"\n{'='*60}")
        print(f"📝 Example {i+1}")
        print(f"{'='*60}")
        print(f"🤖 Generated:\n{candidates[i].strip()}")
        print(f"\n✅ Reference:\n{golds[i].strip()}")
else:
    print("❌ Result files not found. Make sure testing is complete.")

HASIL GENERATED SUMMARIES

📊 Total examples: 1500

📝 Example 1
🤖 Generated:
# person1 # suggests # and # person2 # person2 about the traffic . # person1 ' s car .

✅ Reference:
#Person2# decides to follow #Person1#'s suggestions on quitting driving to work and will try to use public transportations.

📝 Example 2
🤖 Generated:
# person1 # suggests # and # person2 # person2 about the traffic . # person1 ' s car .

✅ Reference:
#Person2# complains to #Person1# about the traffic jam, #Person1# suggests quitting driving and taking public transportation instead.

📝 Example 3
🤖 Generated:
# person2 # tells # person1 # person2 about # person1 . they need person2 # ' t to send .

✅ Reference:
Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.

📝 Example 4
🤖 Generated:
# person1 # suggests # and # person2 # person2 about the traffic . # person1 ' s car .

✅ Reference:
#Person2# arrives

In [28]:
# ============================================================
# FULL SUMMARY - LIHAT HASIL & HITUNG ROUGE
# ============================================================

%cd /content/results

# Rename files
import os
for f in os.listdir('.'):
    if f.startswith('.8000'):
        new_name = f[1:]  # Remove leading dot
        os.rename(f, new_name)

print("="*70)
print("🎉 BERTSUMABS EVALUATION RESULTS")
print("="*70)

# Read files
with open('8000.candidate', 'r') as f:
    candidates = [line.strip() for line in f.readlines() if line.strip()]

with open('8000.gold', 'r') as f:
    references = [line.strip() for line in f.readlines() if line.strip()]

min_len = min(len(candidates), len(references))
print(f"\n📊 Total Test Examples: {min_len}")

# Show examples
print("\n" + "="*70)
print("📝 SAMPLE COMPARISONS")
print("="*70)

for i in range(min(5, min_len)):
    print(f"\n--- Example {i+1} ---")
    print(f"🤖 Generated: {candidates[i][:200]}...")
    print(f"✅ Reference: {references[i][:200]}...")

# Calculate ROUGE
!pip install -q rouge-score
from rouge_score import rouge_scorer
import numpy as np

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

r1_f, r2_f, rL_f = [], [], []
for c, r in zip(candidates[:min_len], references[:min_len]):
    if c and r:
        s = scorer.score(r, c)
        r1_f.append(s['rouge1'].fmeasure)
        r2_f.append(s['rouge2'].fmeasure)
        rL_f.append(s['rougeL'].fmeasure)

print("\n" + "="*70)
print("📊 ROUGE SCORES")
print("="*70)
print(f"""
+----------------+----------+
|     Metric     |    F1    |
+----------------+----------+
|    ROUGE-1     |  {np.mean(r1_f)*100:6.2f}  |
|    ROUGE-2     |  {np.mean(r2_f)*100:6.2f}  |
|    ROUGE-L     |  {np.mean(rL_f)*100:6.2f}  |
+----------------+----------+
""")

print("✅ Evaluation Complete!")

/content/results
🎉 BERTSUMABS EVALUATION RESULTS

📊 Total Test Examples: 1500

📝 SAMPLE COMPARISONS

--- Example 1 ---
🤖 Generated: # person1 # suggests # and # person2 # person2 about the traffic . # person1 ' s car ....
✅ Reference: #Person2# decides to follow #Person1#'s suggestions on quitting driving to work and will try to use public transportations....

--- Example 2 ---
🤖 Generated: # person1 # suggests # and # person2 # person2 about the traffic . # person1 ' s car ....
✅ Reference: #Person2# complains to #Person1# about the traffic jam, #Person1# suggests quitting driving and taking public transportation instead....

--- Example 3 ---
🤖 Generated: # person2 # tells # person1 # person2 about # person1 . they need person2 # ' t to send ....
✅ Reference: Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore....

--- Example 4 ---
🤖 Generated: # person1 # suggests # and # 

In [ ]:
# ============================================================
# CELL 12: SAVE KE GOOGLE DRIVE
# ============================================================

import shutil
import os

print("="*50)
print("SAVING TO GOOGLE DRIVE")
print("="*50)

# Buat folder di Drive
save_dir = '/content/drive/MyDrive/BertSumAbs_DialogSum'
os.makedirs(save_dir, exist_ok=True)

# Copy files
items_to_copy = [
    ('/content/models', 'models'),
    ('/content/results', 'results'),
    ('/content/logs', 'logs'),
    ('/content/bert_data', 'bert_data'),
    ('/content/data_processed', 'data_processed'),
]

for src, dst in items_to_copy:
    dst_path = os.path.join(save_dir, dst)
    if os.path.exists(src):
        if os.path.exists(dst_path):
            shutil.rmtree(dst_path)
        shutil.copytree(src, dst_path)
        print(f"✅ Copied: {src} → {dst_path}")
    else:
        print(f"⚠️ Not found: {src}")

print(f"\n📁 Saved to: {save_dir}")
print("\nContents:")
!ls -la {save_dir}

# BART

In [29]:
# ============================================================
# CELL 1: SETUP BART
# ============================================================

!pip install -q transformers datasets evaluate accelerate

import torch
from transformers import BartForConditionalGeneration, BartTokenizer
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import DataCollatorForSeq2Seq
from datasets import load_dataset
import evaluate
import numpy as np

print("="*60)
print("BART FOR DIALOGUE SUMMARIZATION")
print("="*60)

# Check GPU
print(f"\n🖥️ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Load model dan tokenizer
print("\n📌 Loading BART model...")
model_name = "facebook/bart-base"
model = BartForConditionalGeneration.from_pretrained(model_name)
tokenizer = BartTokenizer.from_pretrained(model_name)

# Load dataset
print("📌 Loading DialogSum dataset...")
dataset = load_dataset('knkarthick/dialogsum')

print(f"\n✅ Model: {model_name}")
print(f"✅ Train: {len(dataset['train'])} examples")
print(f"✅ Val: {len(dataset['validation'])} examples")
print(f"✅ Test: {len(dataset['test'])} examples")

BART FOR DIALOGUE SUMMARIZATION

🖥️ GPU: Tesla T4

📌 Loading BART model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

📌 Loading DialogSum dataset...

✅ Model: facebook/bart-base
✅ Train: 12460 examples
✅ Val: 500 examples
✅ Test: 1500 examples


In [30]:
# ============================================================
# CELL 2: PREPROCESSING
# ============================================================

def preprocess_function(examples):
    # Tokenize input (dialogue)
    inputs = tokenizer(
        examples['dialogue'],
        max_length=1024,
        truncation=True,
        padding='max_length'
    )

    # Tokenize target (summary)
    labels = tokenizer(
        text_target=examples['summary'],
        max_length=128,
        truncation=True,
        padding='max_length'
    )

    inputs['labels'] = labels['input_ids']
    return inputs

print("📌 Preprocessing dataset...")
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset['train'].column_names,
    desc="Tokenizing"
)

print("✅ Preprocessing complete!")
print(f"   Train: {len(tokenized_dataset['train'])}")
print(f"   Val: {len(tokenized_dataset['validation'])}")
print(f"   Test: {len(tokenized_dataset['test'])}")

📌 Preprocessing dataset...


Tokenizing:   0%|          | 0/12460 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/1500 [00:00<?, ? examples/s]

✅ Preprocessing complete!
   Train: 12460
   Val: 500
   Test: 1500


In [31]:
# ============================================================
# CELL 3: TRAINING SETUP
# ============================================================

# Metric
rouge = evaluate.load('rouge')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Decode predictions
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 dengan pad token id
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return {k: round(v * 100, 2) for k, v in result.items()}

# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

# Training arguments - OPTIMIZED untuk hasil bagus
training_args = Seq2SeqTrainingArguments(
    output_dir='./bart-dialogsum',

    # Training params
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch = 16

    # Optimizer
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=500,

    # Logging & Eval
    logging_steps=100,
    eval_strategy='steps',
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,

    # Generation
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,

    # Optimization
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model='rougeL',
    greater_is_better=True,

    # Misc
    report_to='none',
    seed=42,
)

# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("✅ Training setup complete!")
print(f"""
📋 Training Configuration:
   - Epochs: 3
   - Batch size: 4 x 4 = 16 (effective)
   - Learning rate: 5e-5
   - Warmup steps: 500
   - Eval every: 500 steps
""")

/tmp/ipython-input-3918258810.py:73: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


✅ Training setup complete!

📋 Training Configuration:
   - Epochs: 3
   - Batch size: 4 x 4 = 16 (effective)
   - Learning rate: 5e-5
   - Warmup steps: 500
   - Eval every: 500 steps



In [32]:
# ============================================================
# CELL 4: TRAINING
# ============================================================

print("="*60)
print("🚀 STARTING TRAINING")
print("="*60)
print("⏳ Estimated time: 30-45 minutes on Colab GPU\n")

# Train!
trainer.train()

print("\n✅ Training complete!")

🚀 STARTING TRAINING
⏳ Estimated time: 30-45 minutes on Colab GPU



Step,Training Loss,Validation Loss


TypeError: sequence item 32: expected str instance, NoneType found

In [ ]:
# ============================================================
# CELL 5: SAVE MODEL
# ============================================================

# Save model
trainer.save_model('./bart-dialogsum-final')
tokenizer.save_pretrained('./bart-dialogsum-final')

print("✅ Model saved to ./bart-dialogsum-final")

In [ ]:
# ============================================================
# CELL 6: EVALUATE ON TEST SET
# ============================================================

print("="*60)
print("📊 EVALUATING ON TEST SET")
print("="*60)

# Evaluate
results = trainer.evaluate(tokenized_dataset['test'])

print("\n📊 TEST SET RESULTS:")
print("+"*40)
for key, value in results.items():
    if 'rouge' in key.lower() or 'loss' in key.lower():
        print(f"|  {key:20} |  {value:10.2f}  |")
print("+"*40)

In [33]:
# ============================================================
# FULL BART TRAINING - FIXED VERSION
# ============================================================

!pip install -q transformers datasets evaluate accelerate

import torch
import numpy as np
from transformers import BartForConditionalGeneration, BartTokenizer
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import DataCollatorForSeq2Seq
from datasets import load_dataset
import evaluate

print("="*60)
print("BART FOR DIALOGUE SUMMARIZATION (FIXED)")
print("="*60)

# ========================================
# 1. LOAD MODEL & DATA
# ========================================
print("\n📌 Loading model and data...")

model_name = "facebook/bart-base"
model = BartForConditionalGeneration.from_pretrained(model_name)
tokenizer = BartTokenizer.from_pretrained(model_name)
dataset = load_dataset('knkarthick/dialogsum')

print(f"✅ Model: {model_name}")
print(f"✅ Dataset loaded: {len(dataset['train'])} train examples")


BART FOR DIALOGUE SUMMARIZATION (FIXED)

📌 Loading model and data...
✅ Model: facebook/bart-base
✅ Dataset loaded: 12460 train examples


In [34]:
# ========================================
# 2. PREPROCESSING
# ========================================
print("\n📌 Preprocessing...")

def preprocess_function(examples):
    inputs = tokenizer(
        examples['dialogue'],
        max_length=1024,
        truncation=True,
        padding='max_length'
    )

    labels = tokenizer(
        text_target=examples['summary'],
        max_length=128,
        truncation=True,
        padding='max_length'
    )

    inputs['labels'] = labels['input_ids']
    return inputs

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset['train'].column_names,
    desc="Tokenizing"
)

print("✅ Preprocessing complete!")



📌 Preprocessing...


Tokenizing:   0%|          | 0/12460 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/1500 [00:00<?, ? examples/s]

✅ Preprocessing complete!


In [36]:
# ========================================
# 3. METRICS - FIXED VERSION
# ========================================
rouge = evaluate.load('rouge')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Handle invalid token IDs
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    predictions = np.clip(predictions, 0, tokenizer.vocab_size - 1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Decode
    try:
        decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    except Exception as e:
        print(f"Decode error: {e}")
        return {'rouge1': 0, 'rouge2': 0, 'rougeL': 0}

    # Clean up
    decoded_preds = [pred.strip() if pred else "" for pred in decoded_preds]
    decoded_labels = [label.strip() if label else "" for label in decoded_labels]

    # Filter empty
    valid_pairs = [(p, l) for p, l in zip(decoded_preds, decoded_labels) if p and l]
    if not valid_pairs:
        return {'rouge1': 0, 'rouge2': 0, 'rougeL': 0}

    preds, refs = zip(*valid_pairs)

    result = rouge.compute(
        predictions=list(preds),
        references=list(refs),
        use_stemmer=True
    )

    return {k: round(v * 100, 2) for k, v in result.items()}


In [37]:
# ========================================
# 4. TRAINING SETUP
# ========================================
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

training_args = Seq2SeqTrainingArguments(
    output_dir='./bart-dialogsum',

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    # Optimizer
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=500,

    # Logging & Eval
    logging_steps=200,
    eval_strategy='steps',
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,

    # Generation
    predict_with_generate=True,
    generation_max_length=128,

    # Optimization
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model='rougeL',
    greater_is_better=True,

    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("✅ Training setup complete!")

/tmp/ipython-input-85659624.py:44: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


✅ Training setup complete!


In [38]:

# ========================================
# 5. TRAIN
# ========================================
print("\n" + "="*60)
print("🚀 STARTING TRAINING")
print("="*60)
print("⏳ Estimated time: 30-45 minutes\n")

trainer.train()

print("\n✅ Training complete!")


🚀 STARTING TRAINING
⏳ Estimated time: 30-45 minutes



Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
500,0.545500,0.354608,44.520000,19.130000,36.510000,36.500000
1000,0.350400,0.322963,47.210000,21.970000,39.490000,39.460000
1500,0.337500,0.312461,47.890000,23.050000,40.040000,40.020000
2000,0.296000,0.309749,47.440000,22.740000,39.750000,39.760000


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].



✅ Training complete!


In [43]:
# ========================================
# 6. SAVE MODEL
# ========================================
trainer.save_model('./bart-dialogsum-final')
tokenizer.save_pretrained('./bart-dialogsum-final')
print("✅ Model saved!")

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [40]:
# ============================================================
# EVALUATE SETELAH TRAINING
# ============================================================

from transformers import pipeline
from rouge_score import rouge_scorer
import numpy as np

print("="*60)
print("📊 EVALUATING BART MODEL")
print("="*60)

# Load model
summarizer = pipeline(
    'summarization',
    model='./bart-dialogsum-final',
    tokenizer='./bart-dialogsum-final',
    device=0 if torch.cuda.is_available() else -1
)

# Get test data
test_dialogues = dataset['test']['dialogue']
test_references = dataset['test']['summary']

print(f"\n📌 Generating summaries for {len(test_dialogues)} examples...")

# Generate summaries
generated = []
batch_size = 8

for i in range(0, len(test_dialogues), batch_size):
    batch = test_dialogues[i:i+batch_size]
    try:
        outputs = summarizer(
            batch,
            max_length=100,
            min_length=20,
            num_beams=4,
            do_sample=False
        )
        generated.extend([o['summary_text'] for o in outputs])
    except Exception as e:
        print(f"Error at batch {i}: {e}")
        generated.extend([""] * len(batch))

    if (i + batch_size) % 200 == 0:
        print(f"   Processed {min(i+batch_size, len(test_dialogues))}/{len(test_dialogues)}")

print(f"✅ Generated {len(generated)} summaries")

# Calculate ROUGE
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

r1, r2, rL = [], [], []
for gen, ref in zip(generated, test_references):
    if gen and ref:
        scores = scorer.score(ref, gen)
        r1.append(scores['rouge1'].fmeasure)
        r2.append(scores['rouge2'].fmeasure)
        rL.append(scores['rougeL'].fmeasure)

print("\n" + "="*60)
print("📊 ROUGE SCORES")
print("="*60)
print(f"""
+------------------+------------+
|      Metric      |   F1 (%)   |
+------------------+------------+
|     ROUGE-1      |   {np.mean(r1)*100:6.2f}   |
|     ROUGE-2      |   {np.mean(r2)*100:6.2f}   |
|     ROUGE-L      |   {np.mean(rL)*100:6.2f}   |
+------------------+------------+
""")

# Show examples
print("\n📝 SAMPLE OUTPUTS:")
for i in range(5):
    print(f"\n--- Example {i+1} ---")
    print(f"🤖 Generated: {generated[i]}")
    print(f"✅ Reference: {test_references[i]}")

📊 EVALUATING BART MODEL


Device set to use cuda:0
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📌 Generating summaries for 1500 examples...


Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   Processed 200/1500


...

   Processed 1000/1500


Your max_length is set to 100, but your input_length is only 64. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=32)
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
B

   Processed 1200/1500


Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Error at batch 1296: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Error at batch 1304: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Error at batch 1312: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-ap

In [44]:
# ============================================================
# ZERO-SHOT - MODEL SUDAH DILATIH UNTUK DIALOG
# Hasil expected: ROUGE-L ~45-48
# ============================================================

!pip install -q transformers datasets rouge-score

from transformers import pipeline
from datasets import load_dataset
from rouge_score import rouge_scorer
import numpy as np

print("="*60)
print("🚀 ZERO-SHOT EVALUATION (NO TRAINING)")
print("="*60)

# Model yang sudah dilatih khusus untuk dialog summarization
model_name = "philschmid/bart-large-cnn-samsum"

print(f"\n📌 Loading {model_name}...")
summarizer = pipeline(
    "summarization",
    model=model_name,
    device=0,
    # Fix warning dengan hanya pakai max_new_tokens
)

# Load test data
dataset = load_dataset('knkarthick/dialogsum')
test_dialogues = dataset['test']['dialogue']
test_references = dataset['test']['summary']

print(f"📌 Testing on {len(test_dialogues)} examples...\n")

# Generate dengan parameter yang benar (tanpa warning)
generated = []
batch_size = 8

for i in range(0, len(test_dialogues), batch_size):
    batch = test_dialogues[i:i+batch_size]
    try:
        outputs = summarizer(
            batch,
            max_new_tokens=100,  # Ganti max_length dengan max_new_tokens
            min_length=20,
            num_beams=4,
            do_sample=False,
            truncation=True
        )
        generated.extend([o['summary_text'] for o in outputs])
    except Exception as e:
        # Fallback ke CPU jika error
        generated.extend([""] * len(batch))

    if (i + batch_size) % 200 == 0:
        print(f"   Processed {min(i+batch_size, len(test_dialogues))}/{len(test_dialogues)}")

print(f"\n✅ Generated {len(generated)} summaries")

# ROUGE Score
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
r1, r2, rL = [], [], []

for gen, ref in zip(generated, test_references):
    if gen and ref:
        s = scorer.score(ref, gen)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rL.append(s['rougeL'].fmeasure)

print(f"""
{'='*60}
📊 ZERO-SHOT ROUGE SCORES (bart-large-cnn-samsum)
{'='*60}
+------------------+------------+
|      Metric      |   F1 (%)   |
+------------------+------------+
|     ROUGE-1      |   {np.mean(r1)*100:6.2f}   |
|     ROUGE-2      |   {np.mean(r2)*100:6.2f}   |
|     ROUGE-L      |   {np.mean(rL)*100:6.2f}   |
+------------------+------------+
""")

# Sample outputs
print("\n📝 SAMPLE OUTPUTS:")
for i in range(5):
    print(f"\n--- Example {i+1} ---")
    print(f"🤖 Generated: {generated[i]}")
    print(f"✅ Reference: {test_references[i]}")

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
The folder you are executing pip from can no longer be found.
🚀 ZERO-SHOT EVALUATION (NO TRAINING)

📌 Loading philschmid/bart-large-cnn-samsum...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cuda:0


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# ============================================================
# FINE-TUNE DENGAN LoRA - CEPAT & EFISIEN
# ============================================================

!pip install -q transformers datasets peft accelerate

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

print("="*60)
print("🚀 LoRA FINE-TUNING (FAST)")
print("="*60)

# Load pre-trained model untuk dialog
model_name = "philschmid/bart-large-cnn-samsum"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Apply LoRA - hanya train 0.5% parameters!
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Load & preprocess data
dataset = load_dataset('knkarthick/dialogsum')

def preprocess(examples):
    inputs = tokenizer(examples['dialogue'], max_length=512, truncation=True, padding='max_length')
    labels = tokenizer(text_target=examples['summary'], max_length=100, truncation=True, padding='max_length')
    inputs['labels'] = labels['input_ids']
    return inputs

tokenized = dataset.map(preprocess, batched=True, remove_columns=dataset['train'].column_names)

# Fast training config
training_args = Seq2SeqTrainingArguments(
    output_dir='./bart-lora',
    num_train_epochs=1,  # Hanya 1 epoch karena model sudah bagus
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=100,
    logging_steps=50,
    save_steps=500,
    fp16=True,
    eval_strategy='no',
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model),
)

print("\n🚀 Training (~10-15 minutes)...")
trainer.train()

# Save
model.save_pretrained('./bart-lora-final')
tokenizer.save_pretrained('./bart-lora-final')
print("\n✅ Training complete!")

In [ ]:
# Download AMI Corpus
!mkdir -p ../ami_data
%cd ../ami_data

# Download annotations dan transcripts
!wget http://groups.inf.ed.ac.uk/ami/AMICorpusAnnotations/ami_public_manual_1.6.2.zip
!unzip ami_public_manual_1.6.2.zip

# Atau gunakan Hugging Face datasets
!pip install datasets

Streaming output truncated to the last 5000 lines.
  inflating: argumentation/ae/ES2009b.D.argumentstructs.xml  
  inflating: argumentation/ae/IS1007b.C.argumentstructs.xml  
  inflating: argumentation/ae/IS1007d.D.argumentstructs.xml  
  inflating: argumentation/ae/TS3006b.B.argumentstructs.xml  
  inflating: argumentation/ae/ES2003d.C.argumentstructs.xml  
  inflating: argumentation/ae/ES2008b.B.argumentstructs.xml  
  inflating: argumentation/ae/ES2009d.A.argumentstructs.xml  
  inflating: argumentation/ae/ES2007c.A.argumentstructs.xml  
  inflating: argumentation/ae/IS1008d.C.argumentstructs.xml  
  inflating: argumentation/ae/ES2012a.B.argumentstructs.xml  
  inflating: argumentation/ae/ES2012b.A.argumentstructs.xml  
  inflating: argumentation/ae/ES2006d.A.argumentstructs.xml  
  inflating: argumentation/ae/ES2006c.B.argumentstructs.xml  
  inflating: argumentation/ae/TS3003b.C.argumentstructs.xml  
  inflating: argumentation/ae/TS3007b.B.argumentstructs.xml  
  inflating: argume

In [ ]:
from datasets import load_dataset

dialogsum = load_dataset("knkarthick/dialogsum")

print(dialogsum)
print("\nKolom:", dialogsum['train'].column_names)
print("\nContoh:")
print(dialogsum['train'][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

Kolom: ['id', 'dialogue', 'summary', 'topic']

Contoh:
{'id': 'train_0', 'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you s

In [ ]:
from transformers import BertTokenizer
import json
import os

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def preprocess_for_bertsum(dataset, output_dir, split='train'):
    """
    Konversi DialogSum ke format BertSumAbs
    """
    os.makedirs(output_dir, exist_ok=True)

    processed_data = []

    for idx, example in enumerate(dataset[split]):
        try:
            # DialogSum columns: dialogue, summary
            source = example['dialogue']
            target = example['summary']

            # Skip jika kosong
            if not source or not target:
                continue

            # Tokenize
            src_tokens = tokenizer.tokenize(str(source))[:510]
            tgt_tokens = tokenizer.tokenize(str(target))[:128]

            data_point = {
                'src': src_tokens,
                'tgt': tgt_tokens,
                'src_txt': str(source),
                'tgt_txt': str(target)
            }
            processed_data.append(data_point)

        except Exception as e:
            if idx < 5:
                print(f"Error pada index {idx}: {e}")
            continue

    # Save
    output_path = os.path.join(output_dir, f'{split}.json')
    with open(output_path, 'w') as f:
        for item in processed_data:
            f.write(json.dumps(item) + '\n')

    print(f"✓ Saved {len(processed_data)} examples to {output_path}")
    return processed_data

# Jalankan preprocessing
output_dir = '/content/data_processed'

print("=== Preprocessing Dataset ===")
for split in dataset.keys():
    print(f"\nProcessing {split}...")
    preprocess_for_bertsum(dataset, output_dir, split)

# Verifikasi
print("\n=== File yang dibuat ===")
!ls -la /content/data_processed/

=== Preprocessing Dataset ===

Processing train...
✓ Saved 12460 examples to /content/data_processed/train.json

Processing validation...
✓ Saved 500 examples to /content/data_processed/validation.json

Processing test...
✓ Saved 1500 examples to /content/data_processed/test.json

=== File yang dibuat ===
total 37228
drwxr-xr-x 2 root root     4096 Jan 28 17:43 .
drwxr-xr-x 1 root root     4096 Jan 28 17:42 ..
-rw-r--r-- 1 root root  3919087 Jan 28 17:46 test.json
-rw-r--r-- 1 root root 32892167 Jan 28 17:46 train.json
-rw-r--r-- 1 root root  1294692 Jan 28 17:46 validation.json


In [ ]:
# Lihat contoh hasil preprocessing
print("=== Contoh Hasil Preprocessing ===")
!head -n 1 /content/data_processed/train.json | python -m json.tool

=== Contoh Hasil Preprocessing ===
{
    "src": [
        "#",
        "person",
        "##1",
        "#",
        ":",
        "hi",
        ",",
        "mr",
        ".",
        "smith",
        ".",
        "i",
        "'",
        "m",
        "doctor",
        "hawkins",
        ".",
        "why",
        "are",
        "you",
        "here",
        "today",
        "?",
        "#",
        "person",
        "##2",
        "#",
        ":",
        "i",
        "found",
        "it",
        "would",
        "be",
        "a",
        "good",
        "idea",
        "to",
        "get",
        "a",
        "check",
        "-",
        "up",
        ".",
        "#",
        "person",
        "##1",
        "#",
        ":",
        "yes",
        ",",
        "well",
        ",",
        "you",
        "haven",
        "'",
        "t",
        "had",
        "one",
        "for",
        "5",
        "years",
        ".",
        "you",
        "should",
        "have",

In [ ]:
%cd /content

# Clone PreSumm (BertSumAbs)
!git clone https://github.com/nlpyang/PreSumm.git

# Install dependencies
!pip install pytorch-transformers tensorboardX multiprocess pyrouge

print("\n✓ PreSumm repository ready!")
!ls PreSumm/

/content
fatal: destination path 'PreSumm' already exists and is not an empty directory.

✓ PreSumm repository ready!
ami_data   json_data  logs    raw_data	 requirements.txt  src
bert_data  LICENSE    models  README.md  results	   urls


In [ ]:
%cd /content/PreSumm/src

import torch
import json
import os
import gc
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def convert_to_bertsum_format_fixed(input_dir, output_dir, split):
    """
    Konversi JSON ke format PyTorch (.pt) untuk BertSum dengan SEMUA field yang diperlukan
    """
    os.makedirs(output_dir, exist_ok=True)

    input_file = os.path.join(input_dir, f'{split}.json')

    if not os.path.exists(input_file):
        print(f"File {input_file} tidak ditemukan!")
        return

    datasets = []

    with open(input_file, 'r') as f:
        for idx, line in enumerate(f):
            data = json.loads(line.strip())

            src_txt = data['src_txt']
            tgt_txt = data['tgt_txt']

            # Split source menjadi kalimat (per baris atau per speaker turn)
            # Untuk dialog, split berdasarkan newline atau pattern speaker
            src_sentences = [s.strip() for s in src_txt.replace('\r\n', '\n').split('\n') if s.strip()]

            # Jika tidak ada split yang bagus, split per kalimat dengan '.'
            if len(src_sentences) <= 1:
                import re
                src_sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', src_txt) if s.strip()]

            # Minimal 1 kalimat
            if len(src_sentences) == 0:
                src_sentences = [src_txt]

            # Tokenize setiap kalimat
            src_subtokens_all = []
            clss = []
            segs = []
            current_seg = 0

            for i, sent in enumerate(src_sentences):
                # Tambah [CLS] di awal setiap kalimat
                clss.append(len(src_subtokens_all))
                src_subtokens_all.append('[CLS]')

                # Tokenize kalimat
                sent_tokens = tokenizer.tokenize(sent)
                src_subtokens_all.extend(sent_tokens)

                # Tambah [SEP] di akhir kalimat
                src_subtokens_all.append('[SEP]')

                # Segment IDs (alternating 0 dan 1)
                sent_length = len(sent_tokens) + 2  # +2 untuk [CLS] dan [SEP]
                segs.extend([current_seg] * sent_length)
                current_seg = 1 - current_seg  # Toggle antara 0 dan 1

                # Batasi panjang maksimal
                if len(src_subtokens_all) >= 510:
                    src_subtokens_all = src_subtokens_all[:510]
                    segs = segs[:510]
                    break

            # Convert tokens ke IDs
            src_subtoken_ids = tokenizer.convert_tokens_to_ids(src_subtokens_all)

            # Filter clss yang valid (dalam range)
            clss = [c for c in clss if c < len(src_subtoken_ids)]

            # src_sent_labels: untuk abstractive, kita bisa set semua ke 0 atau buat dummy labels
            # Atau set kalimat pertama sebagai "penting"
            src_sent_labels = [0] * len(clss)
            if len(src_sent_labels) > 0:
                src_sent_labels[0] = 1  # Anggap kalimat pertama penting

            # Tokenize target
            tgt_subtokens = ['[unused0]'] + tokenizer.tokenize(tgt_txt)[:126] + ['[unused1]']
            # [unused0] = BOS, [unused1] = EOS untuk BertSumAbs
            tgt_subtoken_ids = tokenizer.convert_tokens_to_ids(tgt_subtokens)

            # Skip jika data tidak valid
            if len(src_subtoken_ids) < 5 or len(tgt_subtoken_ids) < 3:
                continue
            if len(clss) == 0:
                continue

            data_dict = {
                'src': src_subtoken_ids,
                'tgt': tgt_subtoken_ids,
                'segs': segs,
                'clss': clss,
                'src_sent_labels': src_sent_labels,
                'src_txt': src_sentences,
                'tgt_txt': tgt_txt
            }
            datasets.append(data_dict)

            # Progress
            if (idx + 1) % 2000 == 0:
                print(f"  Processed {idx + 1} examples...")

    # Save as .pt file
    output_file = os.path.join(output_dir, f'dialogsum.{split}.pt')
    torch.save(datasets, output_file)
    print(f"✓ Saved {len(datasets)} examples to {output_file}")

    # Clear memory
    gc.collect()

    return len(datasets)

# Hapus file lama jika ada
!rm -rf /content/bert_data
!mkdir -p /content/bert_data

# Konversi semua splits
input_dir = '/content/data_processed'
output_dir = '/content/bert_data'

print("=== Konversi ke Format BertSum (Fixed) ===\n")
for split in ['train', 'validation', 'test']:
    print(f"Converting {split}...")
    convert_to_bertsum_format_fixed(input_dir, output_dir, split)
    print()

print("\n=== File .pt yang dibuat ===")
!ls -la /content/bert_data/

/content/PreSumm/src
=== Konversi ke Format BertSum (Fixed) ===

Converting train...
  Processed 2000 examples...
  Processed 4000 examples...
  Processed 6000 examples...
  Processed 8000 examples...
  Processed 10000 examples...
  Processed 12000 examples...
✓ Saved 12460 examples to /content/bert_data/dialogsum.train.pt

Converting validation...
✓ Saved 500 examples to /content/bert_data/dialogsum.validation.pt

Converting test...
✓ Saved 1500 examples to /content/bert_data/dialogsum.test.pt


=== File .pt yang dibuat ===
total 32224
drwxr-xr-x 2 root root     4096 Jan 28 17:48 .
drwxr-xr-x 1 root root     4096 Jan 28 17:46 ..
-rw-r--r-- 1 root root  3419927 Jan 28 17:48 dialogsum.test.pt
-rw-r--r-- 1 root root 28441245 Jan 28 17:48 dialogsum.train.pt
-rw-r--r-- 1 root root  1123771 Jan 28 17:48 dialogsum.validation.pt


In [ ]:
%cd /content/PreSumm/src

# Backup file original
!cp models/data_loader.py models/data_loader.py.backup

# Patch file dengan sed - ganti operasi boolean yang deprecated
!sed -i 's/1 - (src == 0)/(src != 0).long()/g' models/data_loader.py
!sed -i 's/1 - (tgt == 0)/(tgt != 0).long()/g' models/data_loader.py
!sed -i 's/1 - (segs == 0)/(segs != 0).long()/g' models/data_loader.py

# Verifikasi perubahan
print("=== Verifikasi Patch ===")
!grep -n "!= 0" models/data_loader.py | head -10

/content/PreSumm/src
=== Verifikasi Patch ===
33:            mask_src = (src != 0).long()
34:            mask_tgt = (tgt != 0).long()


In [ ]:
# Lihat baris yang diubah
print("=== Cek baris 30-40 di data_loader.py ===")
!sed -n '30,40p' models/data_loader.py

=== Cek baris 30-40 di data_loader.py ===
            tgt = torch.tensor(self._pad(pre_tgt, 0))

            segs = torch.tensor(self._pad(pre_segs, 0))
            mask_src = (src != 0).long()
            mask_tgt = (tgt != 0).long()


            clss = torch.tensor(self._pad(pre_clss, -1))
            src_sent_labels = torch.tensor(self._pad(pre_src_sent_labels, 0))
            mask_cls = (clss != -1).long()
            clss[clss == -1] = 0


In [ ]:
%cd /content/PreSumm/src

# Fix baris mask_cls
!sed -i 's/1 - (clss == -1)/(clss != -1).long()/g' models/data_loader.py

# Verifikasi
print("=== Verifikasi Patch ===")
!sed -n '30,45p' models/data_loader.py

/content/PreSumm/src
=== Verifikasi Patch ===
            tgt = torch.tensor(self._pad(pre_tgt, 0))

            segs = torch.tensor(self._pad(pre_segs, 0))
            mask_src = (src != 0).long()
            mask_tgt = (tgt != 0).long()


            clss = torch.tensor(self._pad(pre_clss, -1))
            src_sent_labels = torch.tensor(self._pad(pre_src_sent_labels, 0))
            mask_cls = (clss != -1).long()
            clss[clss == -1] = 0
            setattr(self, 'clss', clss.to(device))
            setattr(self, 'mask_cls', mask_cls.to(device))
            setattr(self, 'src_sent_labels', src_sent_labels.to(device))




In [ ]:
import torch

# Load dan cek isi file
print("=== Verifikasi Format Data ===\n")
data = torch.load('/content/bert_data/dialogsum.train.pt')
print(f"Jumlah data: {len(data)}")
print(f"\nKeys dalam data[0]: {list(data[0].keys())}")

print(f"\n=== Detail data[0] ===")
for key, value in data[0].items():
    if isinstance(value, list):
        if len(value) > 10:
            print(f"  {key}: {value[:10]}... (len={len(value)})")
        else:
            print(f"  {key}: {value}")
    else:
        print(f"  {key}: {value}")

# Cek bahwa semua field required ada
required_fields = ['src', 'tgt', 'segs', 'clss', 'src_sent_labels', 'src_txt', 'tgt_txt']
print(f"\n=== Cek Required Fields ===")
for field in required_fields:
    status = "✓" if field in data[0] else "✗"
    print(f"  {status} {field}")

=== Verifikasi Format Data ===

Jumlah data: 12460

Keys dalam data[0]: ['src', 'tgt', 'segs', 'clss', 'src_sent_labels', 'src_txt', 'tgt_txt']

=== Detail data[0] ===
  src: [101, 1001, 2711, 2487, 1001, 1024, 7632, 1010, 2720, 1012]... (len=287)
  tgt: [1, 2720, 1012, 3044, 1005, 1055, 2893, 1037, 4638, 1011]... (len=43)
  segs: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]... (len=287)
  clss: [0, 25, 47, 75, 101, 139, 148, 182, 191, 218]... (len=12)
  src_sent_labels: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]... (len=12)
  src_txt: ["#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?", '#Person2#: I found it would be a good idea to get a check-up.', "#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.", '#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?', '#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.', '#Person2#: Ok.', 

In [ ]:
%cd /content/PreSumm/src

# Pastikan folder ada
!mkdir -p /content/models
!mkdir -p /content/logs

!python train.py \
    -task abs \
    -mode train \
    -bert_data_path /content/bert_data/dialogsum \
    -dec_dropout 0.2 \
    -model_path /content/models \
    -sep_optim true \
    -lr_bert 0.002 \
    -lr_dec 0.2 \
    -save_checkpoint_steps 1000 \
    -batch_size 4 \
    -train_steps 5000 \
    -report_every 100 \
    -accum_count 8 \
    -use_bert_emb true \
    -use_interval true \
    -warmup_steps_bert 500 \
    -warmup_steps_dec 500 \
    -max_pos 512 \
    -visible_gpus 0 \
    -log_file /content/logs/train_abs.log

/content/PreSumm/src
[2026-01-28 17:49:33,141 INFO] Namespace(task='abs', encoder='bert', mode='train', bert_data_path='/content/bert_data/dialogsum', model_path='/content/models', result_path='../results/cnndm', temp_dir='../temp', batch_size=4, test_batch_size=200, max_pos=512, use_interval=True, large=False, load_from_extractive='', sep_optim=True, lr_bert=0.002, lr_dec=0.2, use_bert_emb=True, share_emb=False, finetune_bert=True, dec_dropout=0.2, dec_layers=6, dec_hidden_size=768, dec_heads=8, dec_ff_size=2048, enc_hidden_size=512, enc_ff_size=512, enc_dropout=0.2, enc_layers=6, ext_dropout=0.2, ext_layers=2, ext_hidden_size=768, ext_heads=8, ext_ff_size=2048, label_smoothing=0.1, generator_shard_size=32, alpha=0.6, beam_size=5, min_length=15, max_length=150, max_tgt_len=140, param_init=0, param_init_glorot=True, optim='adam', lr=1, beta1=0.9, beta2=0.999, warmup_steps=8000, warmup_steps_bert=500, warmup_steps_dec=500, max_grad_norm=0, save_checkpoint_steps=1000, accum_count=8, repo

In [ ]:
%cd /content/PreSumm/src

# Buat folder results
!mkdir -p /content/results

# Evaluasi pada validation set
!python train.py \
    -task abs \
    -mode validate \
    -batch_size 4 \
    -test_batch_size 4 \
    -bert_data_path /content/bert_data/dialogsum \
    -log_file /content/logs/val_abs.log \
    -model_path /content/models \
    -sep_optim true \
    -use_interval true \
    -visible_gpus 0 \
    -max_pos 512 \
    -max_length 100 \
    -alpha 0.95 \
    -min_length 20 \
    -result_path /content/results/

/content/PreSumm/src


In [ ]:
%cd /content/PreSumm/src

# Cek checkpoint terbaik
!ls -la /content/models/*.pt

# Gunakan checkpoint terakhir (ganti nama file sesuai yang ada)
# Contoh: model_step_5000.pt atau model_step_1000.pt

In [ ]:
# Generate summary pada test set
# GANTI model_step_5000.pt dengan checkpoint yang ada!

!python train.py \
    -task abs \
    -mode test \
    -batch_size 4 \
    -test_batch_size 4 \
    -bert_data_path /content/bert_data/dialogsum \
    -log_file /content/logs/test_abs.log \
    -test_from /content/models/model_step_5000.pt \
    -sep_optim true \
    -use_interval true \
    -visible_gpus 0 \
    -max_pos 512 \
    -max_length 100 \
    -alpha 0.95 \
    -min_length 20 \
    -result_path /content/results/

In [ ]:
# Lihat file hasil
print("=== File Hasil ===")
!ls -la /content/results/

# Lihat beberapa contoh generated summary
print("\n=== Contoh Generated Summaries ===")
!head -10 /content/results/*.candidate 2>/dev/null || echo "File .candidate tidak ditemukan"

# Lihat reference (gold summary)
print("\n=== Reference Summaries ===")
!head -10 /content/results/*.gold 2>/dev/null || echo "File .gold tidak ditemukan"

In [ ]:
# Install FFmpeg and then reinstall torchcodec
!apt-get update
!apt-get install -y ffmpeg
!pip install --force-reinstall torchcodec

print("FFmpeg and torchcodec installation/reinstallation complete. Please re-run the previous cell to load the AMI dataset.")

In [ ]:
# Load AMI Dataset dari Hugging Face
from datasets import load_dataset

# Load AMI corpus (menggunakan konfigurasi 'ihm' sebagai contoh)
# Ini akan mendefinisikan variabel 'ami_dataset'
ami_dataset = load_dataset("edinburghcstr/ami", "ihm")

# Cek struktur dataset
print("=== AMI Dataset Info ===")
print(ami_dataset)
print("\n--- Contoh data train AMI ---")
print(ami_dataset['train'][0])



In [ ]:
import json
import os
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def preprocess_ami_for_bertsum(dataset, output_dir, split='train'):
    """
    Konversi AMI Corpus ke format BertSumAbs
    """
    os.makedirs(output_dir, exist_ok=True)

    processed_data = []

    for idx, example in enumerate(dataset[split]):
        # Gabungkan transcript dari meeting
        transcript = ' '.join(example['transcript']) if isinstance(example['transcript'], list) else example['transcript']

        # Summary (jika ada)
        summary = example.get('summary', '')

        # Tokenize
        src_tokens = tokenizer.tokenize(transcript)[:510]  # Max length
        tgt_tokens = tokenizer.tokenize(summary)[:128]

        data_point = {
            'src': src_tokens,
            'tgt': tgt_tokens,
            'src_txt': transcript,
            'tgt_txt': summary
        }
        processed_data.append(data_point)

    # Save sebagai JSON
    output_path = os.path.join(output_dir, f'{split}.json')
    with open(output_path, 'w') as f:
        for item in processed_data:
            f.write(json.dumps(item) + '\n')

    print(f"Saved {len(processed_data)} examples to {output_path}")
    return processed_data

# Jalankan preprocessing
output_dir = '/content/ami_processed'
train_data = preprocess_ami_for_bertsum(ami_dataset, output_dir, 'train')
val_data = preprocess_ami_for_bertsum(ami_dataset, output_dir, 'validation')
test_data = preprocess_ami_for_bertsum(ami_dataset, output_dir, 'test')